# Save RR curves as netCDF files

The GBD RR curves can be downloaded [here](https://doi.org/10.6069/vkdr-qy60) in csv format. This script converts them to netCDF.

In [1]:
import os
import re
import glob
import pandas as pd
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Setup ===
DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23" / "RR_curves")

# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

# === Loop through variables ===
for var in health_vars:
    # Searching for files with glob. Not all files have the same date format.
    pattern = os.path.join(DIR, f"IHME_GBD_2023_AIR_POLLUTION_*_PM_RR_{var}_MEAN_*.CSV")
    matches = glob.glob(pattern)

    if len(matches) == 0:
        raise FileNotFoundError(f"No file found for variable: {var}")
    if len(matches) > 1:
        raise ValueError(f"Multiple files matched for variable {var}: {matches}")

    in_file = os.path.basename(matches[0])
    df = pd.read_csv(matches[0])

    # Normalise columns across both formats (GBD2021 vs GBD2023)
    if "risk" in df.columns and "exposure" not in df.columns:
        # New format: no 'cause' column, rename 'risk' -> 'exposure'
        df = df.rename(columns={"risk": "exposure"})
        df.insert(0, "cause", var)   # synthesise cause column so rest of code is unchanged

    # Extract the date range (e.g. 1990_2021) from the input filename
    date_match = re.search(r"AIR_POLLUTION_(\d{4}_\d{4})_PM", in_file)
    if not date_match:
        raise ValueError(f"Could not parse date range from filename: {in_file}")
    date_range = date_match.group(1)   # e.g. "1990_2021" or "1990_2022"

    # set a multi-index exposure
    df_indexed = df.set_index(["exposure"])
    # convert to xarray Dataset
    ds = df_indexed.to_xarray()

    cause_ID = ds.cause[0].values
    ds = ds.drop_vars("cause")
    description = (
        "Global Burden of Disease Collaborative Network. Global Burden of "
        "Disease Study 2023 (GBD 2023) Air Pollution Exposure Estimates "
        "and Risk Curves 1990-2023. Seattle, United States of America: "
        "Institute for Health Metrics and Evaluation (IHME), 2026."
    )
    ds.attrs["cause_ID"] = str(cause_ID)
    ds.attrs["cause_variable"] = var
    ds.attrs["description"] = description

    out_file = f"IHME_GBD_2023_AIR_POLLUTION_{date_range}_PM_RR_{var}.nc"
    out_path = os.path.join(DIR, out_file)
    ds.to_netcdf(out_path)
    print(f"Saved {out_path}")

Saved /glade/work/awells/workflow/GBD23/RR_curves/IHME_GBD_2023_AIR_POLLUTION_1990_2021_PM_RR_COPD.nc
Saved /glade/work/awells/workflow/GBD23/RR_curves/IHME_GBD_2023_AIR_POLLUTION_1990_2021_PM_RR_DIABETES.nc
Saved /glade/work/awells/workflow/GBD23/RR_curves/IHME_GBD_2023_AIR_POLLUTION_1990_2022_PM_RR_ISCHEMIC_HEART_DISEASE.nc
Saved /glade/work/awells/workflow/GBD23/RR_curves/IHME_GBD_2023_AIR_POLLUTION_1990_2022_PM_RR_LOWER_RESPIRATORY_INFECTIONS.nc
Saved /glade/work/awells/workflow/GBD23/RR_curves/IHME_GBD_2023_AIR_POLLUTION_1990_2021_PM_RR_LUNG_CANCER.nc
Saved /glade/work/awells/workflow/GBD23/RR_curves/IHME_GBD_2023_AIR_POLLUTION_1990_2022_PM_RR_STROKE.nc
Saved /glade/work/awells/workflow/GBD23/RR_curves/IHME_GBD_2023_AIR_POLLUTION_1990_2022_PM_RR_DEMENTIA.nc
